# YOLO11n Fine-Tuning for Chess Piece Detection

## Introduction

This notebook presents the fine-tuning of **YOLO11 Nano (YOLO11n)** for chess piece object detection.

The objective is to adapt a lightweight, pretrained YOLO11n model to a custom chess piece detection task. The resulting model will be evaluated not only in terms of detection performance, but also with respect to its suitability for deployment on an embedded computing platform.

The dataset used for this experiment is the **Chess Pieces Object Detection Dataset** published by Roboflow. It contains images of chess boards with bounding-box annotations for twelve different chess piece classes:

* `white-king`
* `white-queen`
* `white-bishop`
* `white-knight`
* `white-rook`
* `white-pawn`
* `black-king`
* `black-queen`
* `black-bishop`
* `black-knight`
* `black-rook`
* `black-pawn`

The original dataset contains 292 images and 2,894 annotated objects. The images were captured from a fixed viewpoint, making the dataset representative of a controlled chessboard detection scenario.

The complete pipeline covered in this notebook consists of:

1. Dataset download
2. Dataset structure inspection
3. Image and annotation validation
4. Class distribution analysis
5. Bounding-box analysis
6. Dataset visualization
7. Dataset preparation
8. YOLO11n fine-tuning
9. Model evaluation
10. Inference on previously unseen images
11. Model export for deployment

The model is initialized from pretrained **YOLO11n** weights rather than trained from scratch. Fine-tuning allows the pretrained visual representations to be adapted to the specific appearance, scale, and spatial arrangement of chess pieces while maintaining the computational efficiency of the Nano architecture.

Particular attention is given to dataset quality and model size because the final model is intended for execution on an embedded platform. Therefore, the training pipeline is designed to preserve a lightweight model while achieving reliable chess piece detection.

The dataset is provided by Roboflow and is released under a **Public Domain** license.


## Environment Setup

The environment is configured with the libraries required for dataset handling, visualization, and YOLO11 fine-tuning.

The main framework used for object detection is **Ultralytics**, which provides the YOLO11 implementation and the training, validation, inference, and export interfaces.

Additional libraries are used for numerical operations, image processing, dataset analysis, and visualization.

The environment is also inspected before starting the experiment to verify the installed versions and the availability of hardware acceleration.


In [ ]:
%pip install -q ultralytics pandas matplotlib seaborn pillow pyyaml

In [ ]:
import os
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import yaml

import torch
import ultralytics

from ultralytics import YOLO

In [3]:
print("=" * 60)
print("Environment Information")
print("=" * 60)

print(f"Python        : {sys.version.split()[0]}")
print(f"Platform      : {platform.platform()}")
print(f"PyTorch       : {torch.__version__}")
print(f"Ultralytics  : {ultralytics.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version  : {torch.version.cuda}")
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
else:
    print("GPU           : CPU")

Environment Information
Python        : 3.12.3
Platform      : Linux-7.0.0-30-generic-x86_64-with-glibc2.39
PyTorch       : 2.13.0+cu130
Ultralytics  : 8.4.131

CUDA available: True
CUDA version  : 13.0
GPU           : NVIDIA GeForce RTX 5090


In [4]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to: {SEED}")

Random seed set to: 42
